# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The data is described using a Croissant schema, allowing for programmatic discovery, access, and processing.

### Dataset Source
The dataset is accessed via the Croissant schema URL and includes multiple record sets and fields. All identifiers for record sets and fields are referenced by their `@id` fields according to the Croissant specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"Dataset: {getattr(dataset.metadata, 'name', '')}\nDescription: {getattr(dataset.metadata, 'description', '')}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', '')}")
print(f"FAIR-compliant citation: {getattr(dataset.metadata, 'citeAs', '')}\n")

## 2. Data Overview
Let's review the available record sets in the dataset and examine their fields and `@id`s. The Croissant specification organizes data tables into *record sets*, which may be joined or analyzed separately.

Below, we list each record set along with its `@id` and fields.

In [ ]:
# Retrieve available record sets
record_sets = list(dataset.metadata.record_sets)
print(f"Record sets (@id): {', '.join([getattr(rs, '@id', '') for rs in record_sets])}\n")

# Display fields and columns for each record set
for rs in record_sets:
    print(f"Record Set: {getattr(rs, '@id', '')}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    print("  Fields:")
    for field in list(rs.fields):
        print(f"    - {getattr(field, '@id', '')} ({getattr(field, 'name', '')})")
    print()

## 3. Data Extraction
We will extract the data from all available record sets into separate pandas DataFrames for analysis. Each record set is referenced by its unique `@id`.

In [ ]:
# Extract tabular data for each record set

# Prepare @id list for record sets
record_set_ids = [getattr(rs, '@id', '') for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set '{record_set_id}': {df.shape[0]} records, {df.shape[1]} columns.")

# Display columns from the first record set as an example
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"\nAvailable columns in '{example_id}':")
    print(dataframes[example_id].columns.tolist())
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's analyze and preprocess the data. We'll:
- Select a numeric field for filtering and normalization,
- Filter records above a threshold,
- Normalize the field,
- Group by a categorical field, if present.

**Note:** All fields and columns are referenced by their `@id` (Croissant identifier) unless otherwise noted. Adjust field IDs according to the data overview above.

In [ ]:
# Choose the main record set for EDA
main_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_record_set_id].copy()
# Display numeric columns to help select one
numeric_cols = df.select_dtypes(include=['number', 'float', 'int']).columns.tolist()
print(f"Available numeric fields (@id): {numeric_cols}")

# Select a numeric field, for example purposes (replace with actual `@id` as desired)
numeric_field_id = numeric_cols[0] if numeric_cols else None
if numeric_field_id is not None:
    # Set threshold for filtering
    threshold = df[numeric_field_id].mean()  # Example: use mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} rows")

    # Normalize the selected numeric field (Z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field if available
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    group_field = categorical_cols[0] if categorical_cols else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"Mean of {numeric_field_id} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print('No numeric fields found for analysis.')

## 5. Visualization

Let's plot the distribution of the selected numeric field, and the group means if grouping was possible.

> Note: Depending on the backend or colab environment, you may need to run `%matplotlib inline`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        grouped_df.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded a FAIR-compliant mass spectrometry dataset described using a Croissant schema,
- Explored record set and field identifiers,
- Extracted and analyzed data with `mlcroissant`,
- Filtered and normalized a numeric field, and visualized its distribution.

Further analyses can be conducted by referencing the full Croissant schema and utilizing all available fields and record sets via their `@id`s.

**For reproducibility and complete analysis, always refer to fields by their Croissant `@id`!**